# Imports

The cell below imports all of the functions and packages that will be used in this lab

In [4]:
from helpers import load_data
import numpy as np

# Load data

The data used in this example was gotten form this page: https://medium.com/data-science/decision-trees-for-classification-complete-example-d0bc17fcf1c2

I have created a helper function, *load_data()*, that will return a 12 x 3 matrix with training examples and a 12 x 1 matrix with corresponding targets

In [2]:
# Import our training set
X_train, y_train = load_data()

# See shape of data
print(f"Training data shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")

# See examples from training set
print(f"First 5 training examples: {X_train[:4]}")
print(f"Labels of first 5 training examples: {y_train[:4]}")

Run this or else
[0 1 1 0 0 1 0 1 0 1]
Training data shape: (10, 3)
Labels shape: (10,)
First 5 training examples: [[24  0  0]
 [30  1  1]
 [36  0  1]
 [36  0  0]]
Labels of first 5 training examples: [0 1 1 0]


# Build Tree
A decision tree is built recursively by passing examples to a node, getting the feature that will best split the examples for a node, visit the left child of the node then visiting the right child of the node. For this scenario, there are 2 stopping criteria:

1. All examples belong to the same target
2. The maximum depth of 10 has been reached

The feature with the best split is the one that has the most information gain. To get the information gain of a particular feature we do the following:

$$
\text{Information gain}
=
H\left(p_1^{\text{root}}\right)
-
\left(
w^{\text{left}} H\left(p_1^{\text{left}}\right)
+
w^{\text{right}} H\left(p_1^{\text{right}}\right)
\right)
$$

This calculates gain of information by the decision node splitting the data into left and right sub trees. When 0 or 1 is passed to H we just return 1.

*p* is the probability of finding 1 in the examples given to a node i.e. the number of examples with output label 1 divided by total number of examples. The *H(p)* function returns the impurity of the examples from a probability. It is calculcated by doing:

$$
\left( -p_1 \log_2(p_1) \right)
-
\left( (1 - p_1)\log_2(1 - p_1) \right)
$$

These measures of impurity are then multiplied by $w^{\text{left}}$ and $w^{\text{right}}$ which is the number of examples given to the left and right child of the node divided by the total number of examples

The root node of the tree is at depth zero. The depth of a child node is equal to depth of the parent + 1

In [14]:
def H(p):
    """
    Returns the measure of impurity from a probability p given to it
    
    Args:
        p (scalar): probability of finding item with target label 1 in examples given to node
        
    Returns:
        H(p) (scalar): measure of impurity
    """
    assert 0 <= p <= 1, "Invalid probability"
    if p == 0 or p == 1:
        return 0
    
    return (-p * np.log2(p)) - ((1 - p) * np.log2(1 - p))

In [15]:
# Test H function. The function called with 0 should return 0, called with 1 should return 1 and called with 1/2 should return 1
print(f"H(0) = {H(0)} which is {H(0) == 0}")
print(f"H(1) = {H(1)} which is {H(1) == 0}")
print(f"H(0.5) = {H(0.5)} which is {H(0.5) == 1}")

H(0) = 0 which is True
H(1) = 0 which is True
H(0.5) = 1.0 which is True


In [16]:
def weighted_impurity(w, p):
    """
    Returns the weighted impurity of an example
    
    Args:
        w (scalar): Weight of example given as ratio of number of items and total examples
        p (scalar): probability of item in example being 1
        
    Returns:
        wH(p): weighted impurity
    """
    H_p = H(p)
    return w * H_p

In [19]:
def information_gain(p_root, p_left, p_right, w_left, w_right):
    """
    Returns information gain of a feature
    
    Args:
        p_root(scalar) : probability of find target output in parent node
        p_left(scalar) : probabilty of finding target output in left child
        p_right(scalar) : probability of finding target output in right child
        w_left(scalar): weight of items in left child as fraction of examples in left child over total examples
        w_right(scalar): weight of items in right child as fraction of examples in right child over total examples
        
    Returns:
        IG(scalar): Information gain
    """
    H_p1_root = H(p_root)
    wHleft = weighted_impurity(w_left, p_left)
    wHright = weighted_impurity(w_right, p_right)
    return H_p1_root - (wHleft + wHright)
    

In [20]:
# Test with example
IG1 = information_gain(0.5, 0.8, 0.2, 0.5, 0.5) # should return around 0.28
print(f"{IG1} should be around 0.28")

0.2780719051126377 should be around 0.28
